Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu prediksi omzet harian tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi total_penjualan sebuah pesanan berdasarkan atribut-atributnya.

In [1]:
# Menyiapkan Dataset
# Jalankan cell berikut untuk membuat dataset.

import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [2]:
# Membuat SparkSession.

import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas11_Naufal_MLlib") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/15 06:00:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/15 06:00:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 4.1.1


In [3]:
# Jmbatan ke Local.

path_csv = "file://" + os.path.abspath("tugas11_penjualan.csv")
df_kopi = spark.read.csv(path_csv, header=True, inferSchema=True)

print("Dataset berhasil dimuat. Total baris:", df_kopi.count())
df_kopi.show(5)

Dataset berhasil dimuat. Total baris: 2500
+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows


A. Feature Engineering (bobot 20%)

Gunakan StringIndexer untuk mengubah kolom nama_menu, cabang, dan waktu_pesan menjadi indeks numerik. Gabungkan seluruh fitur (qty, harga_satuan, dan ketiga kolom hasil indexing) menjadi satu kolom features menggunakan VectorAssembler.B. Train/Test Split & Pelatihan Model (bobot 20%)

Bagi data menjadi 75% training dan 25% testing (seed=101). Latih model LinearRegression dengan labelCol="total_penjualan".

In [4]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. StringIndexer untuk 3 kolom kategorikal.
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

model_idx_menu = indexer_menu.fit(df_kopi)
model_idx_cabang = indexer_cabang.fit(df_kopi)
model_idx_waktu = indexer_waktu.fit(df_kopi)

df_indexed = model_idx_menu.transform(df_kopi)
df_indexed = model_idx_cabang.transform(df_indexed)
df_indexed = model_idx_waktu.transform(df_indexed)

# 2. VectorAssembler untuk menggabungkan seluruh fitur.
assembler_utama = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df_siap = assembler_utama.transform(df_indexed)
df_siap.select("nama_menu", "cabang", "waktu_pesan", "features", "total_penjualan").show(5, truncate=False)

+--------------+----------+-----------+-------------------------+------------------+
|nama_menu     |cabang    |waktu_pesan|features                 |total_penjualan   |
+--------------+----------+-----------+-------------------------+------------------+
|Kentang Goreng|Semarang  |Malam      |[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|Kentang Goreng|Yogyakarta|Sore       |[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|Kentang Goreng|Semarang  |Siang      |[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|Kentang Goreng|Yogyakarta|Pagi       |[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|Roti Bakar    |Magelang  |Malam      |[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+--------------+----------+-----------+-------------------------+------------------+
only showing top 5 rows


B. Train/Test Split & Pelatihan Model (bobot 20%)

Bagi data menjadi 75% training dan 25% testing (seed=101). Latih model LinearRegression dengan labelCol="total_penjualan".

In [5]:
from pyspark.ml.regression import LinearRegression

train_data, test_data = df_siap.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train_data.count()}")
print(f"Jumlah data testing  : {test_data.count()}")

lr_utama = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_utama = lr_utama.fit(train_data)

print("\nModel utama berhasil dilatih!")
print("Koefisien :", model_utama.coefficients)
print("Intercept :", model_utama.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661

Model utama berhasil dilatih!
Koefisien : [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept : -66903.17766485889


C. Evaluasi Model (bobot 20%)

Hitung dan tampilkan RMSE dan R² dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

In [6]:
from pyspark.ml.evaluation import RegressionEvaluator

prediksi_utama = model_utama.transform(test_data)

eval_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
eval_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse_utama = eval_rmse.evaluate(prediksi_utama)
r2_utama = eval_r2.evaluate(prediksi_utama)

print(f"RMSE Model Utama : Rp {rmse_utama:,.2f}")
print(f"R² Model Utama   : {r2_utama:.4f} ({r2_utama*100:.2f}%)")

RMSE Model Utama : Rp 10,900.80
R² Model Utama   : 0.9107 (91.07%)


Model regresi ini sangat baik dan layak digunakan sebagai alat bantu prediksi omzet Kopi Nusantara. Nilai R² yang sangat tinggi (di atas 0,90) menunjukkan bahwa lebih dari 90% variasi total penjualan berhasil dijelaskan oleh fitur-fitur yang ada, sementara nilai rata-rata kesalahan prediksi (RMSE) tergolong sangat kecil dibandingkan nominal transaksi sesungguhnya.   

D. Eksperimen Perbandingan Fitur (bobot 25%)

Latih model kedua yang HANYA menggunakan fitur qty dan harga_satuan (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

In [7]:
assembler_kedua = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_2"
)

train_data_2 = assembler_kedua.transform(train_data)
test_data_2 = assembler_kedua.transform(test_data)

lr_kedua = LinearRegression(featuresCol="features_2", labelCol="total_penjualan")
model_kedua = lr_kedua.fit(train_data_2)

prediksi_kedua = model_kedua.transform(test_data_2)
r2_kedua = eval_r2.evaluate(prediksi_kedua)

print(f"R² Model Utama (5 fitur) : {r2_utama:.4f}")
print(f"R² Model Kedua (2 fitur) : {r2_kedua:.4f}")

R² Model Utama (5 fitur) : 0.9107
R² Model Kedua (2 fitur) : 0.9108


Penambahan informasi nama_menu, cabang, dan waktu_pesan tidak meningkatkan akurasi prediksi secara berarti (tidak terlalu berpengaruh), karena nilai R² antara model utama (5 fitur) dan model kedua (2 fitur) nyaris identik. Hal ini disebabkan oleh hubungan matematis di mana total_penjualan secara langsung ditentukan oleh perkalian qty dan harga_satuan, sehingga atribut cabang maupun waktu pemesanan tidak memberikan pengaruh signifikan terhadap total nominal per transaksi. 

E. Prediksi Skenario Baru (bobot 15%)

Buat sebuah DataFrame baru berisi 3 skenario pesanan (data dummy) (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan StringIndexer dan VectorAssembler yang sama seperti data training — perhatikan: gunakan model StringIndexer yang sama yang sudah di-fit() sebelumnya, jangan membuat StringIndexer baru). Tampilkan hasil prediksi total_penjualan untuk ketiga skenario tersebut.

In [8]:
data_skenario = spark.createDataFrame([
    ("Kopi Susu", "Magelang", "Pagi", 2, 18000),
    ("Cappuccino", "Yogyakarta", "Sore", 4, 25000),
    ("Roti Bakar", "Semarang", "Malam", 3, 20000)
], ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"])

skenario_idx = model_idx_menu.transform(data_skenario)
skenario_idx = model_idx_cabang.transform(skenario_idx)
skenario_idx = model_idx_waktu.transform(skenario_idx)

skenario_siap = assembler_utama.transform(skenario_idx)
hasil_prediksi_baru = model_utama.transform(skenario_siap)

print("=== HASIL PREDIKSI 3 SKENARIO PESANAN BARU ===")
hasil_prediksi_baru.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction"
).show(truncate=False)

=== HASIL PREDIKSI 3 SKENARIO PESANAN BARU ===


+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |2  |18000       |31479.510248098144|
|Cappuccino|Yogyakarta|Sore       |4  |25000       |97908.2695218741  |
|Roti Bakar|Semarang  |Malam      |3  |20000       |60100.58033204226 |
+----------+----------+-----------+---+------------+------------------+



In [9]:
# Penutup.
spark.stop()
print("SparkSession Tugas 11 ditutup.")

SparkSession Tugas 11 ditutup.
